# 🛵 Zepto Delivery Time Prediction & Analysis
**Author: Shreya Wargantiwar**  
**Tools: Python, Pandas, Scikit-learn, Matplotlib, Seaborn**

This notebook analyzes 9,800+ Zepto delivery records to find delay patterns and predict delivery delays using Machine Learning.

## 📂 Step 1: Upload Your CSV File
Run this cell and click the **Choose Files** button to upload your `Zepto_Analysis.csv`

In [ ]:
from google.colab import files
uploaded = files.upload()
print('✅ File uploaded successfully!')

## 📦 Step 2: Install & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported!')

## 🔍 Step 3: Load & Clean Data

In [ ]:
# Load the uploaded CSV
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

# Drop empty columns
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Fix dates
df['Order Date'] = pd.to_datetime(df['Order Date'].str.strip(), format='mixed', dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'].str.strip(), format='mixed', dayfirst=True)

# Extract date features
df['Order_Month'] = df['Order Date'].dt.month
df['Order_DayOfWeek'] = df['Order Date'].dt.dayofweek
df['Order_Year'] = df['Order Date'].dt.year

print(f'✅ Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'\nColumns: {df.columns.tolist()}')
df.head()

## 📊 Step 4: Exploratory Data Analysis

In [ ]:
print('='*50)
print('KEY STATISTICS')
print('='*50)

delay_rate = df['Delivery_Status'].value_counts(normalize=True)['Delayed'] * 100
print(f'\n📦 Total Orders: {len(df):,}')
print(f'⚠️  Overall Delay Rate: {delay_rate:.1f}%')
print(f'⏱  Avg Delivery Time: {df["Delivery_Time_Minutes"].mean():.1f} minutes')
print(f'🔴 Delayed avg time: {df[df["Delivery_Status"]=="Delayed"]["Delivery_Time_Minutes"].mean():.1f} min')
print(f'🟢 On-time avg time: {df[df["Delivery_Status"]=="On-Time"]["Delivery_Time_Minutes"].mean():.1f} min')

print(f'\nDelay Rate by Region:')
region_delay = df.groupby('Region')['Delivery_Status'].apply(
    lambda x: (x == 'Delayed').sum() / len(x) * 100).round(1)
print(region_delay.sort_values(ascending=False))

## 📈 Step 5: Visualizations Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('🛵 Zepto Delivery Analysis Dashboard', fontsize=16, fontweight='bold')

# Plot 1: Delivery Status Pie
status_counts = df['Delivery_Status'].value_counts()
axes[0,0].pie(status_counts.values, labels=status_counts.index,
              autopct='%1.1f%%', colors=['#2ecc71','#e74c3c'], startangle=90)
axes[0,0].set_title('On-Time vs Delayed Deliveries')

# Plot 2: Delivery Time Distribution
axes[0,1].hist(df[df['Delivery_Status']=='On-Time']['Delivery_Time_Minutes'],
               alpha=0.7, label='On-Time', color='#2ecc71', bins=20)
axes[0,1].hist(df[df['Delivery_Status']=='Delayed']['Delivery_Time_Minutes'],
               alpha=0.7, label='Delayed', color='#e74c3c', bins=20)
axes[0,1].set_title('Delivery Time Distribution')
axes[0,1].set_xlabel('Minutes')
axes[0,1].legend()

# Plot 3: Region Delay Rate
region_delay = df.groupby('Region')['Delivery_Status'].apply(
    lambda x: (x=='Delayed').sum()/len(x)*100).sort_values()
region_delay.plot(kind='barh', ax=axes[0,2], color='#3498db')
axes[0,2].set_title('Delay Rate by Region (%)')

# Plot 4: Delivery Type Delay
delivery_delay = df.groupby('Delivery_Type')['Delivery_Status'].apply(
    lambda x: (x=='Delayed').sum()/len(x)*100).sort_values(ascending=False)
delivery_delay.plot(kind='bar', ax=axes[1,0], color='#9b59b6', rot=15)
axes[1,0].set_title('Delay Rate by Delivery Type (%)')

# Plot 5: Monthly Orders
df.groupby('Order_Month').size().plot(kind='line', ax=axes[1,1],
    marker='o', color='#e67e22', linewidth=2)
axes[1,1].set_title('Monthly Order Volume')
axes[1,1].set_xlabel('Month')

# Plot 6: Customer Type Delay
customer_delay = df.groupby('Customer_Type')['Delivery_Status'].apply(
    lambda x: (x=='Delayed').sum()/len(x)*100).sort_values(ascending=False)
customer_delay.plot(kind='bar', ax=axes[1,2], color='#1abc9c', rot=0)
axes[1,2].set_title('Delay Rate by Customer Type (%)')

plt.tight_layout()
plt.savefig('zepto_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard created!')

## 🤖 Step 6: Machine Learning — Predict Delivery Delays

In [ ]:
# Feature Engineering
features = ['Delivery_Type', 'Customer_Type', 'Region', 'Product_Category',
            'Delivery_Time_Minutes', 'Order_Value', 'Sales',
            'Order_Month', 'Order_DayOfWeek']

df_ml = df[features + ['Delivery_Status']].copy()

le = LabelEncoder()
for col in ['Delivery_Type', 'Customer_Type', 'Region', 'Product_Category']:
    df_ml[col] = le.fit_transform(df_ml[col])

df_ml['Target'] = (df_ml['Delivery_Status'] == 'Delayed').astype(int)
df_ml = df_ml.drop('Delivery_Status', axis=1)

# Train/Test Split
X = df_ml[features]
y = df_ml['Target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Results
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred) * 100
print(f'✅ Model Accuracy: {accuracy:.1f}%')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['On-Time', 'Delayed']))

## 🔑 Step 7: Feature Importance — What Causes Delays?

In [ ]:
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(10,6))
importances.plot(kind='bar', color='#3498db')
plt.title('What Causes Delivery Delays?', fontsize=13)
plt.ylabel('Importance Score')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print('\nTop 5 delay factors:')
for feat, imp in importances.head(5).items():
    print(f'  {feat}: {imp:.3f}')

## 💾 Step 8: Download Your Charts

In [ ]:
files.download('zepto_dashboard.png')
files.download('feature_importance.png')
print('✅ Charts downloaded to your computer!')

## 📊 Final Summary
| Metric | Value |
|---|---|
| Total Orders | 9,800 |
| Overall Delay Rate | 43.8% |
| Avg Delivery Time | 19 minutes |
| ML Model Accuracy | 100% |
| Top Delay Factor | Delivery Time Minutes |

**Author: Shreya Wargantiwar | MCA Data Science | Sri Balaji University, Pune**